In [ ]:
#| hide
from segmentation_tools_pt.core import *
from segmentation_tools_pt.channels import *

# segmentation_tools_pt

> A modular PyTorch toolkit for image segmentation: build multi-channel inputs, cut patches, train, evaluate and export. Generic across **multi-channel** inputs and **multi-category** masks, with binary pin-detection as the worked example.

Everything is authored in notebooks under `nbs/` and exported to `src/segmentation_tools_pt/` by nbdev. Each module does one job, every function ships with an example, and the test cells double as the test suite.

## Modules

| Notebook | Module | Responsibility |
|---|---|---|
| `00_core` | `core` | seeding, device, `SegConfig` |
| `01_channels` | `channels` | build multi-channel inputs from source images |
| `02_masks` | `masks` | clean and encode masks (binary + multi-class) |
| `03_patches` | `patches` | sliding-window patch extraction |
| `04_dataset_build` | `dataset_build` | source-level split + parallel build |
| `05_transforms` | `transforms` | augmentation + per-channel stats |
| `06_dataset` | `dataset` | dataset, sampling weights, dataloaders |
| `07_losses` | `losses` | Tversky/Dice/Boundary, combined `SegLoss` |
| `08_metrics` | `metrics` | pixel + object-level metrics |
| `09_model` | `model` | Any smp architecture + encoder; freeze/LR helpers |
| `10_train` | `train` | epoch steps, checkpointing, train loop, callbacks |
| `11_inference` | `inference` | predict, decode, threshold search |
| `12_visualize` | `visualize` | overlays, sanity checks, top losses |
| `13_export` | `export` | ONNX export + verification |

## Install

```sh
pip install -e .
```

The library runs on Linux (e.g. an HPC Jupyter server). Edit notebooks under `nbs/`, then regenerate the package and run the tests:

```sh
nbdev_export   # notebooks -> src/segmentation_tools_pt/
nbdev_test     # run every notebook's test cells
```

## How to use

Building a 3-channel input from two captures and describing the task is a one-liner each:

In [ ]:
import numpy as np

cfg = SegConfig(in_channels=3, num_classes=1)   # binary task
ch0 = np.zeros((8, 8), np.uint8)
ch1 = np.full((8, 8), 255, np.uint8)
image = build_image([ch0, ch1], ['ch0', 'ch1', 'product'])
image.shape, cfg.is_binary

### End-to-end (binary pin-detection)

The full pipeline, from raw image pairs to an exported model:

```python
from segmentation_tools_pt.patches import PatchConfig
from segmentation_tools_pt.dataset_build import build_patch_dataset
from segmentation_tools_pt.transforms import compute_dataset_stats
from segmentation_tools_pt.model import get_model
from segmentation_tools_pt.train import train
from segmentation_tools_pt.export import export_onnx

# 1. Cut overlapping patches from full images (single-channel: images/{id}.png / masks/{id}.png; multi-channel: images/{id}_ch0.png, _ch1.png)
build_patch_dataset('data/raw', 'data/patches', cfg=PatchConfig(), n_channels=2, num_classes=1)

# 2. Train a U-Net (in_channels = number of built channels, here 3)
model = get_model('mobilenet_v2', in_channels=3, num_classes=1, pretrained=True)
train('data/patches', model, num_classes=1, num_epochs=60, batch_size=16)

# 3. Export for deployment
export_onnx(model, 'model.onnx', in_channels=3, input_size=256)
```

For a **multi-class** task, set `num_classes=K` everywhere and provide integer-label masks (`0` = background, `1..K` = classes). No other changes are needed.

Pass optional **callbacks** (`TrainCallback` subclasses) to `train(..., callbacks=[...])` for early stopping, custom logging, or per-batch inspection. Details: [AGENTS.md](AGENTS.md).

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Testing Copilot skills

If you add a skill under `.github/skills/<name>/SKILL.md`, run the smoke-test notebook to catch common structural mistakes before committing:

```bash
jupyter notebook .github/skills/skill-smoke-test/notebooks/test_skills.ipynb
```

It checks YAML frontmatter, name/folder matching, description keywords, broken local references, and emits copy-pasteable chat prompts to verify the skill actually loads. See `tutorials/03_testing_skills.ipynb` for a step-by-step guide.
